In [ ]:
import sys, os
print("Python :", sys.executable)
print("Cwd    :", os.getcwd())

In [ ]:
# --- Setup : force le cwd à la racine du repo ---
import os
from pathlib import Path

current = Path.cwd()
while current != current.parent and not (current / ".env").exists():
    current = current.parent

if (current / ".env").exists():
    os.chdir(current)
    print(f"✅ cwd : {os.getcwd()}")
    print(f"✅ .env trouvé : {current / '.env'}")
else:
    raise RuntimeError(
        "Impossible de trouver .env. Vérifie qu'il est à la racine du repo."
    )

In [ ]:
%pip install pandas numpy matplotlib seaborn scikit-learn sqlalchemy psycopg2-binary

In [ ]:
%pip install python-dotenv

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

# Charger le fichier .env
load_dotenv()

# Récupérer l'URL complète
DATABASE_URL = os.getenv("NEON_DB_URL")

# Créer le moteur SQLAlchemy
engine = create_engine(DATABASE_URL)

# Charger la table training_vigne
query = "SELECT * FROM training_vigne ORDER BY code_departement, year;"
df = pd.read_sql(query, engine)

df = pd.read_sql("SELECT * FROM training_vigne", engine)
print(f"Avant nettoyage : {df.shape}")

# Suppression des lignes sans rendement enregistré (départements sans tradition viticole reconnue)
df = df.dropna(subset=['rendement_q_ha'])
print(f"Après nettoyage : {df.shape}")

print(df.shape)
df.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
rendement_moyen = df.groupby('year')['rendement_q_ha'].mean()

plt.figure(figsize=(10, 5))
plt.plot(rendement_moyen.index, rendement_moyen.values, marker='o', color='darkred')
plt.title('Rendement moyen national - Vigne (2010-2024)')
plt.xlabel('Année')
plt.ylabel('Rendement (q/ha)')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
cols_num = ['rendement_q_ha', 'temp_saison', 'precip_cumul', 'solar_moy']
plt.figure(figsize=(8, 6))
sns.heatmap(df[cols_num].corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Matrice de corrélation - Vigne')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df['rendement_q_ha'], bins=30, kde=True, color='darkred')
plt.title('Distribution du rendement - Vigne (2010-2024)')
plt.xlabel('Rendement (q/ha)')
plt.show()

In [ ]:
df['periode'] = df['year'].apply(lambda x: '2010-2019' if x < 2020 else '2020-2024')
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='periode', y='rendement_q_ha', palette='Set2')
plt.title('Rendement par période')
plt.xlabel('')
plt.show()
df.drop('periode', axis=1, inplace=True)

In [ ]:
# Séparation temporelle
train = df[df['year'] <= 2020].copy()
test  = df[df['year'] >= 2021].copy()

print(f"Train : {train.shape[0]} lignes ({train['year'].min()}-{train['year'].max()})")
print(f"Test  : {test.shape[0]} lignes ({test['year'].min()}-{test['year'].max()})")

In [ ]:
features = ['temp_saison', 'precip_cumul', 'solar_moy',
            'soil_family_dominant', 'soil_type_dominant', 'year', 'code_departement']
target = 'rendement_q_ha'

X_train = train[features]
y_train = train[target]
X_test  = test[features]
y_test  = test[target]

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

categorical_cols = ['soil_family_dominant', 'soil_type_dominant', 'code_departement']
numerical_cols   = ['temp_saison', 'precip_cumul', 'solar_moy', 'year']

preprocessor = ColumnTransformer([
    ('num', 'passthrough', numerical_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
])

X_train_enc = preprocessor.fit_transform(X_train)
X_test_enc  = preprocessor.transform(X_test)

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# Moyenne par département sur le train
train_avg = train.groupby('code_departement')['rendement_q_ha'].mean().to_dict()
# Moyenne nationale (tous départements confondus) sur le train
national_mean = train['rendement_q_ha'].mean()

# Application de la prédiction avec remplissage par la moyenne nationale
test['pred_baseline'] = test['code_departement'].map(train_avg).fillna(national_mean)

r2_base   = r2_score(y_test, test['pred_baseline'])
rmse_base = np.sqrt(mean_squared_error(y_test, test['pred_baseline']))

print(f"Baseline  R² : {r2_base:.3f} | RMSE : {rmse_base:.1f} q/ha")

In [ ]:
from sklearn.linear_model import RidgeCV

ridge = RidgeCV(alphas=[0.01, 0.1, 1, 10, 100], cv=5)
ridge.fit(X_train_enc, y_train)
y_pred_ridge = ridge.predict(X_test_enc)

r2_ridge   = r2_score(y_test, y_pred_ridge)
rmse_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))

print(f"Ridge     R² : {r2_ridge:.3f} | RMSE : {rmse_ridge:.1f} q/ha")

In [ ]:
%pip install xgboost

In [ ]:
from xgboost import XGBRegressor

xgb = XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42)
xgb.fit(X_train_enc, y_train)
y_pred_xgb = xgb.predict(X_test_enc)

r2_xgb   = r2_score(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))

print(f"XGBoost   R² : {r2_xgb:.3f} | RMSE : {rmse_xgb:.1f} q/ha")

In [ ]:
results = pd.DataFrame({
    'Modèle':   ['Baseline (moy. dép.)', 'Ridge', 'XGBoost'],
    'R² test':  [r2_base, r2_ridge, r2_xgb],
    'RMSE (q/ha)': [rmse_base, rmse_ridge, rmse_xgb]
})
print(results)

In [ ]:
"""
Baseline (moy. dép.)
Référence : la simple moyenne historique par département capture déjà 47,2 % de la variance. Cela montre que le terroir (département) explique une part stable du rendement.

Ridge (linéaire régularisé)
C’est le meilleur modèle. Il améliore nettement la baseline (+1,8 points de R²) en intégrant les variables climatiques et la tendance temporelle. Il explique 49 % de la variance du rendement, ce qui est très honorable pour des données agricoles réelles.

XGBoost
Il est moins bon que Ridge, probablement parce qu’il est plus complexe et que les données ne contiennent pas de fortes non-linéarités.
"""

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred_ridge, alpha=0.5, color='steelblue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--')
plt.xlabel('Rendement observé (q/ha)')
plt.ylabel('Rendement prédit par Ridge (q/ha)')
plt.title('Ridge : Observé vs Prédit (test 2021-2024)')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
""" 
Observé vs Prédit

Alignement global : La majorité des points suit la tendance de la ligne idéale (y = x), confirmant la capacité de Ridge à capturer la tendance générale (R² = 0,49).
Surestimation des bas rendements : Le modèle ne prédit aucune valeur négative, mais surestime systématiquement les rendements faibles (pour un observé < 30 q/ha, les prédictions restent au-dessus de 20-30 q/ha).
Sous-estimation des rendements extrêmes : Pour les rendements observés > 100 q/ha (et l'extrême à > 200 q/ha), le modèle plafonne nettement autour de 75-100 q/ha. La régularisation Ridge ainsi que la linéarité du modèle lissent fortement ces valeurs extrêmes.
"""

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import RidgeCV
import pandas as pd
import numpy as np

# Re-préparer les données avec standardisation des variables numériques
preprocessor_ridge = ColumnTransformer([
    ('num', StandardScaler(), numerical_cols),
    ('cat', 'passthrough', categorical_cols)  # On laisse l'encodage one-hot déjà fait
])
# Mais on ne peut pas appliquer OneHotEncoder en même temps, donc on refait l'encodage complet avec StandardScaler
categorical_cols = ['soil_family_dominant', 'soil_type_dominant', 'code_departement']
numerical_cols   = ['temp_saison', 'precip_cumul', 'solar_moy', 'year']

preprocessor_ridge = ColumnTransformer([
    ('num', StandardScaler(), numerical_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
])

X_train_enc_ridge = preprocessor_ridge.fit_transform(X_train)
X_test_enc_ridge  = preprocessor_ridge.transform(X_test)

# Entraîner Ridge sur ces données standardisées
ridge = RidgeCV(alphas=[0.01, 0.1, 1, 10, 100], cv=5)
ridge.fit(X_train_enc_ridge, y_train)
y_pred_ridge_std = ridge.predict(X_test_enc_ridge)

r2_ridge_std = r2_score(y_test, y_pred_ridge_std)
rmse_ridge_std = np.sqrt(mean_squared_error(y_test, y_pred_ridge_std))
print(f"Ridge standardisé R² : {r2_ridge_std:.3f}, RMSE : {rmse_ridge_std:.1f} q/ha")

In [ ]:
# Récupérer tous les noms de features après encodage
cat_feature_names = preprocessor_ridge.named_transformers_['cat'].get_feature_names_out(categorical_cols)
all_features = numerical_cols + list(cat_feature_names)

coefficients = ridge.coef_
# Trier par valeur absolue décroissante
idx = np.argsort(np.abs(coefficients))[::-1][:15]

plt.figure(figsize=(10, 6))
colors = ['darkred' if coefficients[i] < 0 else 'darkgreen' for i in idx]
plt.barh(range(len(idx)), coefficients[idx], color=colors)
plt.yticks(range(len(idx)), [all_features[i] for i in idx])
plt.xlabel('Coefficient standardisé')
plt.title('Top 15 des coefficients Ridge')
plt.axvline(0, color='black', linewidth=0.8)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
"""
Top 15 des coefficients Ridge

Dominance absolue de la géographie (code_departement) : Le modèle repose quasi exclusivement sur la variable catégorielle du département (ex. 17, 16 et 02 tirent fortement vers le haut ; 06 tire le plus vers le bas).
Impact des sols : Un seul type de sol apparaît dans le top 15 (soil_family_dominant_luvisols, avec un effet négatif).
Absence des variables climatiques et temporelles : Aucune variable climatique (temp_saison, precip_cumul, solar_moy) ni l'année n'apparaît dans le top 15. Le modèle prédit principalement un niveau moyen par département (effet fixe spatial) plutôt qu'une vraie réponse climatique dynamique.
"""

In [ ]:
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)

In [ ]:
residus = y_test - y_pred_ridge_std
plt.figure(figsize=(10, 5))
sns.boxplot(x=test['year'], y=residus, palette='Set2')
plt.axhline(0, color='black', linestyle='--')
plt.title('Résidus (observé - prédit) par année - Ridge')
plt.ylabel('Erreur (q/ha)')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
"""
Résidus par année (2021-2024)

Léger biais selon l'année : Bien que les médianes restent globalement autour de la ligne zéro pour 2022 et 2023, elles sont légèrement négatives en 2021 et 2024, traduisant une sur-estimation moyenne du modèle sur ces deux campagnes.
Variabilité variable (hétéroscédasticité) : L'amplitude des erreurs est nettement plus marquée en 2022 et 2023 qu'en 2021 et 2024, montrant que le modèle a plus de mal à capturer la variabilité de certaines années.
Présence de sous-estimations ponctuelles fortes : Quelques valeurs abérrantes majeures apparaissent vers le haut (notamment un point en 2023 à plus de 140 q/ha), montrant que le modèle rate certains rendements exceptionnellement élevés.
"""

In [ ]:
"""
1. Forces exploitables du modèle
R² = 0,49 : c’est un résultat solide, surtout comparé à la baseline (0,47). R² = 0,49 : un gain modeste mais réel par rapport à la baseline (0,47). L'écart s'est resserré après intégration du Languedoc-Roussillon, dont le climat méditerranéen ajoute de la variance que le modèle peine à capturer aussi bien que sur le reste du vignoble français.
Résidus globalement centrés mais hétérogènes selon l'année : légère sur-estimation en 2021 et 2024, variabilité plus marquée en 2022-2023. Le modèle reste globalement fiable en moyenne, mais sa précision n'est pas uniforme dans le temps.
Interprétabilité : les coefficients Ridge révèlent que l’effet terroir (département) domine. C’est une information métier importante : les différences de rendement entre régions viticoles sont structurelles et stables, même sous un climat changeant.

2. Limites à garder en tête
Prédictions négatives : inacceptables en valeur absolue. Pour la projection future, il faudra clipper les prédictions à zéro (ou à un minimum physiologique) avant de les présenter.
Sous-estimation des extrêmes : le modèle lisse les très bonnes et très mauvaises années. Cela signifie que les projections futures donneront une tendance moyenne mais ne captureront pas les millésimes extrêmes.
Absence des variables climatiques dans le top 15 : le modèle repose surtout sur la moyenne historique départementale. La sensibilité au climat est faible. Cela peut être dû à une forte variabilité interannuelle non expliquée par les simples moyennes saisonnières.
Hétéroscédasticité : la variabilité des erreurs diffère selon les années (plus forte en 2022-2023), ce qui questionne la fiabilité uniforme de l'intervalle de confiance utilisé pour la projection future (± RMSE global, qui suppose implicitement une erreur homogène).

3. Conséquences pour la projection future
Le modèle peut être utilisé tel quel pour estimer des rendements futurs moyens, à condition de bien expliquer ses limites.
Les projections seront conservatrices : elles refléteront surtout l’effet département, avec une modulation modeste par le climat futur.
Si le climat futur dépasse les bornes historiques, le modèle risque d’extrapoler de manière linéaire (ex. températures plus élevées → rendement plus élevé jusqu’à un certain point, puis peut-être une baisse non captée). Il faudra l’interpréter avec prudence.
"""

In [ ]:
# ============================================================
# 6. PROJECTION FUTURE — Rendement estimé sous climat CORDEX (2054, 2084, 2100)
# ============================================================

query_future = """
WITH climate_agg_future AS (
    SELECT coordinates_id, year,
           AVG(temp_mean_future) AS temp_saison,
           SUM(precip_future)    AS precip_cumul,
           AVG(solar_rad_future) AS solar_moy
    FROM src_climate_future
    WHERE month >= 4 AND month <= 10
    GROUP BY coordinates_id, year
),
poids_surface AS (
    SELECT
        CASE
            WHEN code_departement ~ '^0[0-9]{2}$' AND length(code_departement) = 3 THEN right(code_departement, 2)
            WHEN code_departement ~ '^0[0-9]$' THEN code_departement
            WHEN code_departement ~ '^[0-9]+$' AND length(code_departement) <= 2 THEN lpad(code_departement, 2, '0')
            WHEN code_departement ~ '^[0-9]+\\.0$' THEN lpad(replace(code_departement, '.0', ''), 2, '0')
            WHEN code_departement IN ('2A', '02A') THEN '2A'
            WHEN code_departement IN ('2B', '02B') THEN '2B'
            ELSE code_departement
        END AS code_departement,
        coordinates_id,
        surface_ha_12_5km
    FROM src_yields_cordex_2010_2025
    WHERE year = (SELECT MAX(year) FROM src_yields_cordex_2010_2025)
      AND culture_id = 1
      AND surface_ha_12_5km > 0
),
climat_futur_dept AS (
    SELECT
        ps.code_departement,
        ca.year,
        SUM(ca.temp_saison  * ps.surface_ha_12_5km) / NULLIF(SUM(ps.surface_ha_12_5km), 0) AS temp_saison,
        SUM(ca.precip_cumul * ps.surface_ha_12_5km) / NULLIF(SUM(ps.surface_ha_12_5km), 0) AS precip_cumul,
        SUM(ca.solar_moy    * ps.surface_ha_12_5km) / NULLIF(SUM(ps.surface_ha_12_5km), 0) AS solar_moy
    FROM climate_agg_future ca
    JOIN poids_surface ps ON ca.coordinates_id = ps.coordinates_id
    GROUP BY ps.code_departement, ca.year
)
SELECT
    cf.code_departement,
    cf.year,
    cf.temp_saison,
    cf.precip_cumul,
    cf.solar_moy,
    sd.soil_family_dominant,
    sd.soil_type_dominant
FROM climat_futur_dept cf
JOIN sol_dept sd ON cf.code_departement = sd.department_code
ORDER BY cf.code_departement, cf.year;
"""

future_df = pd.read_sql(query_future, engine)

print(future_df.shape)  # attendu : ~92 départements x 3 horizons ≈ 276 lignes
future_df.head()

In [ ]:
# --- Contrôle rapide : pas de valeurs manquantes, départements cohérents avec le train ---
print(future_df.isna().sum())

deps_train  = set(train['code_departement'].unique())
deps_future = set(future_df['code_departement'].unique())
print("Absents en future :", deps_train - deps_future)
print("Nouveaux en future (ignorés par l'encodeur) :", deps_future - deps_train)

In [ ]:
# --- Encodage (transform uniquement, jamais fit_transform) ---
X_future = future_df[features]
X_future_enc_ridge = preprocessor_ridge.transform(X_future)
X_future_enc_xgb   = preprocessor.transform(X_future)

In [ ]:
# --- 6.4 (révisé) : XGBoost uniquement pour la projection future ---
# Ridge écarté : extrapolation non physique confirmée par le diagnostic (valeurs majoritairement négatives)

pred_xgb_future = xgb.predict(X_future_enc_xgb)

future_df['pred_xgb'] = np.clip(pred_xgb_future, 0, None)

future_df['rendement_bas']   = np.clip(future_df['pred_xgb'] - rmse_xgb, 0, None)
future_df['rendement_haut']  = future_df['pred_xgb'] + rmse_xgb
future_df['rendement_moyen'] = future_df['pred_xgb']

future_df[['code_departement', 'year', 'rendement_bas', 'rendement_moyen', 'rendement_haut']].head(10)

In [ ]:
# --- Synthèse nationale par horizon ---
synthese = future_df.groupby('year')[['rendement_bas', 'rendement_moyen', 'rendement_haut']].mean().round(1)
print(synthese)

In [ ]:
# --- Sauvegarde ---
future_df.to_sql('gold_projection_rendement_vigne', engine, if_exists='replace', index=False)

In [ ]:
# Combien de prédictions Ridge étaient négatives AVANT clipping, par horizon ?
future_df['pred_ridge_brut'] = ridge.predict(X_future_enc_ridge)
future_df['pred_xgb_brut']   = xgb.predict(X_future_enc_xgb)

print(future_df.groupby('year')[['pred_ridge_brut', 'pred_xgb_brut']].describe().T)

In [ ]:
future_df.groupby('year')[['temp_saison', 'precip_cumul', 'solar_moy']].mean().round(1)

## Synthèse — Projection du rendement viticole sous climat futur (2054, 2084, 2100)

**Objectif** : estimer l'évolution du rendement de la vigne en France sous trois horizons climatiques (CORDEX), à partir du modèle XGBoost entraîné sur les données historiques 2010-2024.

**Résultats principaux**

| Horizon | Rendement moyen national (q/ha) | Fourchette indicative |
|---|---|---|
| 2054 | 58,0 | 34,9 – 81,2 |
| 2084 | 56,9 | 33,7 – 80,0 |
| 2100 | 61,7 | 38,6 – 84,9 |

**Trois enseignements clés**

- Le rendement projeté reste globalement dans l'ordre de grandeur observé historiquement — pas d'effondrement viticole généralisé à ces horizons.
- **2084 marque un creux**, lié à un pic de précipitations projeté (740 mm contre ~520-630 mm aux autres horizons) plutôt qu'à une sécheresse — le changement climatique redistribue les risques (excès d'eau, pression fongique) plus qu'il ne dégrade uniformément les conditions.
- **14 départements sans tradition viticole** (Bretagne, Normandie, Hauts-de-France, Île-de-France) apparaissent climatiquement compatibles dans la projection — une piste exploratoire pour de futures zones de production, à confirmer par une expertise terrain.

**Méthode retenue** : seul XGBoost a été utilisé pour la projection (Ridge écarté après diagnostic d'instabilité en extrapolation hors de son domaine d'entraînement, notamment sur les valeurs basses). La fourchette reflète l'erreur moyenne du modèle sur le jeu de test.

## Synthèse détaillée — Projection du rendement viticole sous climat futur (2054, 2084, 2100)

### 1. Objectif et périmètre

Étendre le modèle de prédiction du rendement viticole (entraîné sur `training_vigne`, 2010-2024) aux données climatiques CORDEX projetées pour 2054, 2084 et 2100, afin d'estimer l'évolution du rendement et l'émergence de nouvelles zones climatiquement viables.

### 2. Construction du jeu de données de projection

- **Source climatique** : `src_climate_future` (données mensuelles brutes par maille CORDEX), agrégées selon la même méthode que l'historique : moyenne pour `temp_saison` et `solar_moy`, somme pour `precip_cumul`, sur la période avril-octobre (mois 4 à 10), reproduisant exactement la logique de la vue `climate_maille_vigne`.
- **Pondération spatiale** : les agrégats par maille sont pondérés par `surface_ha_12_5km` (2024, dernière année disponible) pour obtenir une moyenne départementale représentative du poids réel du vignoble, à l'identique de la construction de `training_vigne`. La surface est figée sur 2024 : l'hypothèse retenue est une empreinte viticole stable, seul l'effet climatique est projeté.
- **Données pédologiques** : `soil_family_dominant` et `soil_type_dominant` proviennent de la vue `sol_dept` (mode statistique par département), invariantes dans le temps.
- **Résultat** : 92 départements × 3 horizons = **276 lignes**, dont **14 départements sans historique viticole** (22, 08, 27, 29, 35, 50, 53, 56, 60, 61, 80, 90, 91, 95) — non vus par l'encodeur catégoriel à l'entraînement.

### 3. Choix méthodologique : abandon de Ridge pour la projection

Un diagnostic préalable comparant les prédictions brutes (avant clipping) des deux modèles a révélé un problème de stabilité :

| | Ridge (brut) | XGBoost (brut) |
|---|---|---|
| Moyenne 2054 | 33,2 | 58,0 |
| Moyenne 2084 | 19,7 | 56,9 |
| Moyenne 2100 | 24,2 | 61,7 |

Ridge présente une moyenne positive à tous les horizons, mais reste marqué par une forte variabilité et des minimums encore négatifs par endroits (ex. -16,5 en 2084), révélant une instabilité en extrapolation hors du domaine d'entraînement (climat 2010-2020). XGBoost, structurellement borné par les valeurs observées à l'entraînement, reste plus stable et dans une plage crédible (35 à 105 q/ha environ) sur les trois horizons.

**Décision** : seul XGBoost est retenu pour la projection. La fourchette de restitution est calculée comme `prédiction ± RMSE du modèle sur le jeu de test`, plutôt que par la fourchette Ridge/XGBoost initialement envisagée (option écartée après diagnostic).

### 4. Résultats par horizon

| Horizon | Rendement moyen (q/ha) | Borne basse | Borne haute |
|---|---|---|---|
| 2054 | 58,0 | 34,9 | 81,2 |
| 2084 | 56,9 | 33,7 | 80,0 |
| 2100 | 61,7 | 38,6 | 84,9 |

### 5. Analyse du creux 2084 : un mécanisme climatique, pas une dégradation uniforme

| Horizon | temp_saison (°C) | precip_cumul (mm) | solar_moy |
|---|---|---|---|
| 2054 | 16,4 | 520,4 | 225,6 |
| 2084 | 16,6 | **739,9** | 215,1 |
| 2100 | 18,6 | 659,9 | 223,3 |

Le creux de rendement en 2084 coïncide avec un pic net de précipitations (+40 % environ par rapport aux autres horizons) et une baisse du rayonnement solaire — cohérence physique attendue (davantage de nébulosité associée à l'excès pluviométrique). Un excès d'eau pendant la saison de croissance favorise la pression fongique (mildiou, botrytis) et peut diluer la maturité du raisin, mécanismes distincts d'un stress hydrique par sécheresse. Ce résultat illustre que le changement climatique redistribue les facteurs de risque viticole plutôt qu'il ne produit une dégradation monotone dans le temps — la température suit une trajectoire plus linéaire, mais les précipitations montrent une variabilité plus marquée sur le siècle.

### 6. Limites à mentionner

- **`year` comme feature** : l'hypothèse d'une tendance temporelle linéaire au-delà de la plage d'entraînement (2010-2024, extrapolée jusqu'à 2100) reste une hypothèse forte, assumée en cohérence avec la quasi-linéarité des trajectoires climatiques CORDEX elles-mêmes sur cet horizon.
- **Largeur d'intervalle constante** : la fourchette (± RMSE global) est identique pour tous les départements et horizons ; elle sous-estime probablement l'incertitude réelle, d'autant que les résidus du modèle se sont révélés hétéroscédastiques (variabilité plus marquée certaines années) — l'hypothèse d'une erreur homogène est donc une simplification à assumer explicitement.
- **Zones climatiquement émergentes** : les prédictions pour les 14 départements sans historique reposent uniquement sur l'effet climat + sol appris ailleurs, sans effet "terroir" local — à interpréter comme un indice exploratoire, pas comme une prédiction de rendement au même niveau de confiance que les zones viticoles historiques.
- **Surface figée sur 2024** : l'évolution possible du vignoble (extension, recul, replantation) n'est pas modélisée ; seule la réponse climatique de la surface actuelle est projetée.

### Le mot de la fin : pas de chute de rendement prédite...

Le modèle ne prédit pas de baisse marquée du rendement parce qu'il n'a jamais appris de lien fort entre climat et rendement dans les données historiques — il repose principalement sur l'effet terroir. De plus, XGBoost est structurellement incapable d'extrapoler en dessous des pires années connues. Ce résultat n'est donc pas une prédiction de résilience du vignoble français, mais plutôt le reflet d'une limite méthodologique : le modèle ne capture ni les événements climatiques extrêmes, ni les seuils de rupture (stress thermique, sécheresse aiguë) qui sont pourtant les vrais leviers de risque en agriculture face au changement climatique."

In [ ]:
import joblib
from pathlib import Path

# Racine du repo = deux niveaux au-dessus du dossier du notebook (train/notebooks/)
REPO_ROOT = Path.cwd()
MODELS_DIR = REPO_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

# 1. Préprocesseur (utilisé pour XGBoost, sans StandardScaler)
joblib.dump(preprocessor, MODELS_DIR / "preprocessor.joblib")

# 2. Modèle XGBoost entraîné
joblib.dump(xgb, MODELS_DIR / "xgb_model.joblib")

# 3. Liste des variables (pour s'assurer que l'API utilise les mêmes)
joblib.dump(features, MODELS_DIR / "features.joblib")

print(f"✅ Modèle, préprocesseur et features sauvegardés dans {MODELS_DIR}")